# Get NISAR GSLC Image Chips per ISMN Station

<br>

For each ISMN soil moisture station in `ismn_data/spain/stations.geojson`, **processed
independently of the others**, this notebook saves a small pixel window of NISAR L-band
(frequency A) **GSLC** (Geocoded Single-Look Complex) data around the station, for every
acquisition covering it — one NetCDF file per acquisition per station, rather than the
single scalar-per-acquisition CSV rows written by
[`02_get_NISAR_data.ipynb`](02_get_NISAR_data.ipynb) for GCOV.

Each saved file contains, for whichever polarizations the granule has (HH/HV/VH/VV):
the complex SLC window split into `<pol>_real`/`<pol>_imag` (NetCDF has no native complex
type), plus per-pixel geometry cubes covering the same window
(`incidence_angle_deg`, `los_unit_vector_x`/`y`, `local_incidence_angle_deg`) and the
`mask` layer if present, with station/orbit/acquisition metadata as file attributes.
Filenames encode the station name, acquisition datetime, track number, orbit direction,
and the full product native-id, e.g.:

`{station_name}_{YYYYMMDDTHHMMSS}_track{NNN}_{ASC|DESC}_{native_id}.nc`

under `nisar_data/spain_GSLC_windows/`.

It reuses **Option 2 (stream over HTTPS)**, the same pattern as the GCOV notebook: each
granule is opened remotely with `fsspec` + `h5py` — no full-granule downloads.

:::{note} This pipeline's HDF5-layout assumptions have already been checked against real data
The extraction logic lives in
[`util/nisar_gslc_windows.py`](../util/nisar_gslc_windows.py), originally written by
analogy to the verified GCOV pipeline in
[`util/nisar_station_timeseries.py`](../util/nisar_station_timeseries.py) (product-version
dedup, track filtering, and DEM-terrain/local-incidence-angle math are reused directly,
since GSLC shares the same native-id token structure and the same terrain math), before
any direct `h5py`-based GSLC read existed elsewhere in this repo to copy from. Its
assumptions about GSLC's internal layout (`HH`/`HV` as complex64 datasets under
`frequencyA`, a `mask` layer, `radarGrid` metadata matching GCOV's fields, `projection`'s
EPSG code) were since confirmed against a real downloaded Austria GSLC granule — both
the structure (Section 2 below) and the actual saved pixel values (bit-exact match against
the full product, geometry matching an independent DEM-based recomputation to 5 decimal
places).

**Section 2 below still runs the same structure check against a real granule for this
AOI** — cheap, and a reasonable smoke test before Section 4's full batch run, even though
the underlying schema itself is not expected to vary by AOI.
:::

<hr>

## Overview

1. Load station locations
2. Sanity-check GSLC file structure against a real granule
3. Compute local terrain slope/aspect per station from Copernicus GLO-30 (helper, reused from the GCOV notebook)
4. Process each station independently: search, extract window, save
5. (Optional) Visualize combined granule coverage across all stations

<hr>

In [1]:
import sys
import os
import json
from pathlib import Path
from datetime import datetime, timezone

from ipyleaflet import CircleMarker
import earthaccess

REPO_ROOT = os.path.dirname(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from util.aoi_coverage import plot_aoi_coverage
from util.nisar_station_timeseries import compute_station_terrain, get_fsspec_https_session_with_timeout
from util.nisar_gslc_windows import (
    GSLC_SHORT_NAME,
    DEFAULT_WINDOW_FOOTPRINT_M,
    inspect_gslc_structure,
    process_station_gslc,
)

c:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\pyproj\network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


## 1. Load station locations

Same station list and `TEMPORAL` window as
[`02_get_NISAR_data.ipynb`](02_get_NISAR_data.ipynb) — see that notebook's
Section 1 for why `elevation_m` here is ISMN metadata kept only for reference, not what's
actually used for height interpolation later (that's `dem_elevation_m`, read from the
Copernicus DEM in Section 3 below).

`WINDOW_FOOTPRINT_M` is the target window size **in meters on the ground**, not pixels —
chosen to match the HyP3 Sentinel-1 burst product's pixel size (80m, the `20x4` looks
default) so the two sensors' windows cover comparable ground footprints. Since GSLC's
native grid is anisotropic (~10m range x ~5m azimuth) and each granule reports its own
pixel spacing, the actual x/y pixel counts are derived per granule from
`WINDOW_FOOTPRINT_M` rather than being a single fixed pixel count (see
`extract_and_save_gslc_window` in `util/nisar_gslc_windows.py`) — x and y are each rounded
and centered independently, so a window can't be evenly centered on a single pixel with an
even side length either way.

In [2]:
# This notebook was originally written for the Spain AOI; the actual target has
# since moved to Alaska (same fix already applied in 02_get_NISAR_data.ipynb) --
# ismn_data/spain/ doesn't exist, only ismn_data/alaska/ does.
stations_path = Path.cwd() / "ismn_data" / "alaska" / "stations.geojson"

with open(stations_path) as f:
    stations_geojson = json.load(f)

stations = [
    {
        "name": feat["properties"]["station_val"],
        "lon": feat["geometry"]["coordinates"][0],
        "lat": feat["geometry"]["coordinates"][1],
        "elevation_m": float(feat["properties"]["elevation_val"]),
    }
    for feat in stations_geojson["features"]
]

TEMPORAL = ("2026-06-15", datetime.now(timezone.utc).strftime("%Y-%m-%d"))
WINDOW_FOOTPRINT_M = DEFAULT_WINDOW_FOOTPRINT_M  # meters on the ground; see util/nisar_gslc_windows.py

print(f"Loaded {len(stations)} stations")
stations[:3]

Loaded 17 stations


[{'name': 'MooseInc', 'lon': -151.3872, 'lat': 59.68255, 'elevation_m': 37.0},
 {'name': 'AtigunPass',
  'lon': -149.47817,
  'lat': 68.12983,
  'elevation_m': 1457.0},
 {'name': 'Coldfoot', 'lon': -150.183, 'lat': 67.25333, 'elevation_m': 326.0}]

## 2. Sanity-check GSLC file structure against a real granule

Opens one real GSLC granule (the first candidate found for the first station) and prints
its `frequencyA`/`radarGrid`/`identification` layout via `inspect_gslc_structure()`. As
noted at the top of this notebook, this schema has already been confirmed against a real
downloaded Austria GSLC granule; this is a quick smoke test that this AOI's data looks the
same before running Section 4's full batch across all stations.

In [3]:
earthaccess.login()

# Was earthaccess.get_fsspec_https_session() -- that builds its session with no
# timeout at all, so a stalled fetch hangs indefinitely instead of failing and
# retrying (same issue diagnosed and fixed in 02_get_NISAR_data.ipynb; see
# get_fsspec_https_session_with_timeout's docstring in
# util/nisar_station_timeseries.py for the full story).
#
# Only used directly here for the structure sanity-check below -- process_station_gslc()
# now opens its own fresh session per granule instead of sharing this one across the
# whole batch (see get_fsspec_https_session_with_timeout's skip_instance_cache note:
# sharing one session across granules, combined with a timed-out extraction's thread
# being left running in the background rather than cancelled, is what caused saved
# window files' acquisition_datetime_utc/track_number/orbit_pass_direction attrs to
# end up naming a different granule than that same file's own native_id/pixel data).
fs = get_fsspec_https_session_with_timeout()
fsspec_config = {
    "cache_type": "background",
    "block_size": 8 * 1024 * 1024,  # 8 MB
}

_sanity_results = earthaccess.search_data(
    short_name=GSLC_SHORT_NAME,
    point=(stations[0]["lon"], stations[0]["lat"]),
    temporal=TEMPORAL,
)
print(f"Found {len(_sanity_results)} candidate GSLC granule(s) for station '{stations[0]['name']}'")

if _sanity_results:
    inspect_gslc_structure(_sanity_results[0], fs, fsspec_config)
else:
    print("No candidates found — check GSLC_SHORT_NAME in util/nisar_gslc_windows.py against a live CMR search.")

c:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


Found 19 candidate GSLC granule(s) for station 'MooseInc'
native-id: NISAR_L2_PR_GSLC_023_078_A_032_4005_DHDH_A_20260619T143045_20260619T143121_P05023_N_F_J_001

Top-level groups: ['GSLC', 'identification']

/science/LSAR/GSLC/grids/frequencyA contents:
  HH: shape=(76320, 76464) dtype=complex64
  HV: shape=(76320, 76464) dtype=complex64
  azimuthBandwidth: shape=() dtype=float64
  centerFrequency: shape=() dtype=float64
  inputDataExceptionMask: shape=(76320, 76464) dtype=uint8
  listOfPolarizations: shape=(2,) dtype=|S2
  mask: shape=(76320, 76464) dtype=uint8
  numberOfSubSwaths: shape=() dtype=uint8
  projection: shape=() dtype=uint32
  rangeBandwidth: shape=() dtype=float64
  slantRangeSpacing: shape=() dtype=float64
  xCoordinateSpacing: shape=() dtype=float64
  xCoordinates: shape=(76464,) dtype=float64
  yCoordinateSpacing: shape=() dtype=float64
  yCoordinates: shape=(76320,) dtype=float64
  zeroDopplerTimeSpacing: shape=() dtype=float64
  projection raw value: 32605, attrs: {

## 3. Compute local terrain slope/aspect per station from Copernicus GLO-30

Identical to Section 2 of
[`02_get_NISAR_data.ipynb`](02_get_NISAR_data.ipynb) — reuses
`compute_station_terrain()` from `util/nisar_station_timeseries.py` directly, since the
local-incidence-angle math (DEM slope/aspect → surface normal, combined per-acquisition
with the granule's own line-of-sight vector) doesn't depend on GCOV vs. GSLC. See that
notebook's Section 2 for the method and why the DEM-derived `dem_elevation_m` is used
for height interpolation instead of the ISMN-reported elevation.

Each window's `local_incidence_angle_deg` cube (Section 4) uses this single
per-station surface normal for every pixel in the window — terrain slope/aspect isn't
recomputed per pixel, since it varies negligibly across a chip this size (tens of meters).

In [4]:
compute_station_terrain(stations)

Computing local terrain slope/aspect from Copernicus GLO-30 for each station...
Done. 15 unique DEM tiles used for 17 stations.
Slope range across stations: 0.80 - 9.41 deg (median 2.59 deg)


## 4. Process each station independently: search, extract window, save

Implemented by `process_station_gslc()` in
[`util/nisar_gslc_windows.py`](../util/nisar_gslc_windows.py) (which internally calls
`extract_and_save_gslc_window()` per granule). For each station, in turn:

1. **Search** `GSLC_SHORT_NAME` using `point=(lon, lat)`, same as the GCOV pipeline.
2. **Deduplicate** reprocessed CRID versions with `keep_latest_versions` (reused from
   `nisar_station_timeseries.py`, with a GSLC-specific native-id regex).
3. **Filter** to tracks with at least `MIN_ACQUISITIONS_PER_TRACK` acquisitions with
   `filter_tracks_with_min_acquisitions` (same reuse).
4. **Skip granules already saved** — matched by checking whether the granule's native-id
   appears as a substring of any existing filename for this station in `output_dir` (no
   network calls needed for granules already covered).
5. For each remaining (new) granule, **open it remotely** with `fsspec` + `h5py`, under a
   `PER_GRANULE_TIMEOUT_SECONDS` wall-clock deadline (opening a remote GSLC file needs many
   small, sequential HDF5 metadata reads that can add up to minutes for some granules; a
   granule that exceeds the deadline is skipped and retried on the next run — see
   `util/nisar_station_timeseries.py`), and:
   - Locate the nearest frequency A pixel to the station, then a window of pixels around it
     covering approximately `WINDOW_FOOTPRINT_M x WINDOW_FOOTPRINT_M` on the ground (x/y
     pixel counts derived from that granule's own reported spacing, clipped to whatever
     part falls inside the grid extent for a station near an edge).
   - Read whichever of HH/HV/VH/VV are present in the granule, as complex windows split
     into `<pol>_real`/`<pol>_imag` (NetCDF has no native complex type).
   - Read the `mask` window, if present.
   - Compute a **per-pixel** geometry cube across the whole window — `incidence_angle_deg`
     (ellipsoidal), `los_unit_vector_x`/`y`, and `local_incidence_angle_deg` (combining
     each pixel's own line-of-sight vector with the station's single DEM-derived surface
     normal from Section 3) — reusing the same nearest-cell/height-interpolation approach
     as the GCOV pipeline's radarGrid lookups, memoized per unique radarGrid cell so a
     window that maps to only one or two cells on radarGrid's coarser spacing doesn't
     redundantly re-fetch the same values.
   - Save everything to one NetCDF file, named
     `{station_name}_{datetime}_track{NNN}_{ASC|DESC}_{native_id}.nc`, with station/orbit/
     acquisition metadata as file attributes (not the granule's signed download URL, since
     that can carry a temporary access token that shouldn't end up embedded in a saved file).
6. **Move to the next station.** Unlike the GCOV pipeline there's no same-pass merging
   step here — each acquisition (even same-pass adjacent frames) is saved as its own file,
   since merging pixel windows the way scalar dB values are merged doesn't apply.

In [5]:
# This notebook was originally written for the Spain AOI; the actual target has
# since moved to Alaska (same fix already applied in 02_get_NISAR_data.ipynb) --
# fixed here so results don't land mislabeled under nisar_data/spain_GSLC_windows/.
output_dir = Path.cwd() / "nisar_data" / "alaska_GSLC_windows"
output_dir.mkdir(parents=True, exist_ok=True)

all_gpolygons = {}  # native-id -> GPolygon, collected for the optional overview map below

for station in stations:
    process_station_gslc(
        station, output_dir, TEMPORAL, fsspec_config,
        window_footprint_m=WINDOW_FOOTPRINT_M, all_gpolygons=all_gpolygons,
    )

MooseInc: 19 granule(s) after version dedup and track filtering — 0 saved, 19 already saved, 0 skipped (no usable data for this station), 0 timed out


c:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


AtigunPass: 19 granule(s) after version dedup and track filtering — 0 saved, 19 already saved, 0 skipped (no usable data for this station), 0 timed out


c:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


Coldfoot: 20 granule(s) after version dedup and track filtering — 0 saved, 20 already saved, 0 skipped (no usable data for this station), 0 timed out


...

c:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


TurnagainPass: 24 granule(s) after version dedup and track filtering — 0 saved, 24 already saved, 0 skipped (no usable data for this station), 0 timed out


c:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


UpperNomeCreek: 38 granule(s) after version dedup and track filtering — 0 saved, 38 already saved, 0 skipped (no usable data for this station), 0 timed out
Kenai-29-ENE: 24 granule(s) after version dedup and track filtering — 0 saved, 24 already saved, 0 skipped (no usable data for this station), 0 timed out


c:\Users\geoka\miniconda3\envs\nisar-era5\Lib\site-packages\earthaccess\results.py:343: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


## 5. (Optional) Visualize combined granule coverage across all stations

A single overview map built from every unique granule footprint encountered across all per-station searches above, reusing `plot_aoi_coverage` from `util/aoi_coverage.py`.

In [6]:
lons = [s["lon"] for s in stations]
lats = [s["lat"] for s in stations]
pad = 0.05

aoi_wkt = (
    "POLYGON(("
    f"{min(lons) - pad} {min(lats) - pad}, {max(lons) + pad} {min(lats) - pad}, "
    f"{max(lons) + pad} {max(lats) + pad}, {min(lons) - pad} {max(lats) + pad}, "
    f"{min(lons) - pad} {min(lats) - pad}"
    "))"
)

m = plot_aoi_coverage(list(all_gpolygons.values()), aoi_wkt, "GSLC")

for s in stations:
    m.add(CircleMarker(
        location=(s["lat"], s["lon"]),
        radius=4,
        color="red",
        fill_color="red",
        fill_opacity=0.9,
    ))

m

Map(center=[41.570265, 1.1064349999999998], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_i…

<hr>

## Summary

You now have one NetCDF file per GSLC acquisition per station in
`nisar_data/alaska_GSLC_windows/`, named
`{station_name}_{datetime}_track{NNN}_{ASC|DESC}_{native_id}.nc`. Each file holds a
window of pixels centered on that station's nearest GSLC pixel, sized to cover
approximately `WINDOW_FOOTPRINT_M x WINDOW_FOOTPRINT_M` on the ground (default 80x80m,
matching the HyP3 Sentinel-1 burst product's pixel size) — the x/y pixel counts are
derived per granule from that granule's own reported spacing, so they can differ slightly
between granules, with:

- `<pol>_real`/`<pol>_imag` for every polarization the granule has (HH/HV/VH/VV)
- `mask`, if present in the source granule
- `incidence_angle_deg`, `los_unit_vector_x`/`y`, `local_incidence_angle_deg` — per-pixel
  geometry cubes covering the window, the last one using the station's DEM-derived
  surface normal (Section 3)
- file attrs: `station_name`, `station_lon`/`lat`, `epsg`, `acquisition_datetime_utc`,
  `orbit_pass_direction`, `track_number`, `native_id`, `polarizations`,
  `window_footprint_m_requested`, `window_size_x_px`/`window_size_y_px`,
  `window_footprint_x_m_actual`/`window_footprint_y_m_actual` (the real achieved
  footprint, since rounding to a whole pixel count means it won't be exactly
  `WINDOW_FOOTPRINT_M`), `center_row_in_grid`/`center_col_in_grid`

Reload with `xr.open_dataset(path)`; reconstruct a polarization's complex values with
`ds["HH_real"].values + 1j * ds["HH_imag"].values`.

Re-running this notebook after `TEMPORAL`'s end date moves forward only fetches
newly-acquired granules — already-saved ones are skipped by matching native-id against
existing filenames, no network calls needed.

<hr>